---
title: "State and Graph Control"
draft: true
categories: [agents, workflows, langgraph]
---


LangGraph starts with state, nodes, and transitions. The design work happens before any model call: durable facts belong in state, injected clients belong in runtime context, and every route must correspond to a named invariant.

## A minimal typed graph

The first graph makes the review requirement structural. A gap check may revisit collection once, while verification uses `Command` to update the status and choose either export or terminal failure.


In [1]:
from typing import Literal, TypedDict
from IPython.display import Markdown, display
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command

class State(TypedDict, total=False):
    question: str
    plan: list[str]
    evidence: list[str]
    draft: str
    reviewed: bool
    attempts: int
    status: str

def intake(state: State): return {"question": state["question"].strip(), "attempts": 0}
def plan(state: State): return {"plan": ["security", "performance", "operations"]}
def collect(state: State):
    attempt = state.get("attempts", 0) + 1
    evidence = ["audit logs", "pilot thresholds"] if attempt == 1 else ["audit logs", "pilot thresholds", "capacity test"]
    return {"attempts": attempt, "evidence": evidence}
def gap_route(state: State) -> Literal["collect", "draft"]:
    return "draft" if len(state["evidence"]) == 3 else "collect"
def draft(state: State): return {"draft": "; ".join(state["evidence"])}
def review_stub(state: State): return {"reviewed": True}
def verify(state: State) -> Command[Literal["export", "__end__"]]:
    if not state.get("reviewed"):
        return Command(update={"status": "failed: review skipped"}, goto=END)
    return Command(update={"status": "verified"}, goto="export")
def export(state: State): return {"status": "complete"}

builder = StateGraph(State)
for node in (intake, plan, collect, draft, review_stub, verify, export):
    builder.add_node(node)
builder.add_edge(START, "intake")
builder.add_edge("intake", "plan")
builder.add_edge("plan", "collect")
builder.add_conditional_edges("collect", gap_route)
builder.add_edge("draft", "review_stub")
builder.add_edge("review_stub", "verify")
builder.add_edge("export", END)
graph = builder.compile()
display(Markdown("```mermaid\n" + graph.get_graph().draw_mermaid() + "\n```"))


```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	intake(intake)
	plan(plan)
	collect(collect)
	draft(draft)
	review_stub(review_stub)
	verify(verify)
	export(export)
	__end__([<p>__end__</p>]):::last
	__start__ --> intake;
	collect -.-> draft;
	draft --> review_stub;
	intake --> plan;
	plan --> collect;
	review_stub --> verify;
	verify -.-> __end__;
	verify -.-> export;
	export --> __end__;
	collect -.-> collect;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

The cycle is visible rather than buried in a loop. It is also bounded by state: the second collection produces the missing operational fact and routes forward.

## State exposes hidden failures

Invoke the graph and then remove contract fields one at a time. The prose can still look plausible after an ablation, but the verifier can no longer establish why the run is complete.


In [2]:
result = graph.invoke({"question": " Should we adopt AtlasVector? "})
print({key: result[key] for key in ("plan", "evidence", "attempts", "reviewed", "status")})

def missing_invariant(state: State) -> str:
    if not state.get("evidence"): return "evidence provenance is unverifiable"
    if not state.get("reviewed"): return "review compliance is unverifiable"
    if not state.get("status"): return "termination is unverifiable"
    return "contract intact"

ablations = {key: missing_invariant({k: v for k, v in result.items() if k != key})
             for key in ("evidence", "reviewed", "status")}
print(ablations)
assert result["status"] == "complete" and result["attempts"] == 2
assert all(value != "contract intact" for value in ablations.values())


{'plan': ['security', 'performance', 'operations'], 'evidence': ['audit logs', 'pilot thresholds', 'capacity test'], 'attempts': 2, 'reviewed': True, 'status': 'complete'}
{'evidence': 'evidence provenance is unverifiable', 'reviewed': 'review compliance is unverifiable', 'status': 'termination is unverifiable'}


Typed state does not make an answer true. It makes the conditions for accepting that answer observable. Chapter 03 strengthens the weakest part of this prototype: the strings in `evidence` become provenance-preserving domain objects.
